# C2.9 · Case study — Moltbook: 770,000 agents behind one missing policy

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Both directions*

Builds on **[C2.8 · Case study — the Hugging Face / OpenAI agent-swarm incident](https://spbreed.github.io/cyber-commons/lessons/C2.8.html)**.

| | |
|---|---|
| Tools used | Supabase, PostgREST |

## What this lesson is

**What it covers.** Run the same query with and without a row policy, then work out which of the leaked things the platform could actually revoke.

**Why a security engineer needs it.** The blast radius was not the platform's. What leaked were credentials in five other providers' accounts, and the platform could revoke none of them. The control it builds is: row-level policies, credentials out of client-readable tables, and an admin plane the client cannot reach — the controls of A3.8, arriving at a database.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A social network whose members were AI agents shipped its database key to every browser, which is normal, and left row-level security off, which is not. One query returned every agent's record — including the OpenAI, Anthropic and AWS keys of the people who created them, in plaintext.

> **At CyberTravels.** A platform whose members were agents shipped its database key to every browser and left row-level security off. CyberTravels' vector store and CRM sit behind the same kind of API.

## 2 · The framework

```
   browser                    Supabase Data API           agents table
   +-----------+              +-----------------+         +---------------+
   | anon key  | -----------> |  row-level      |  ...    | id            |
   | (by       |              |  security       |         | owner         |
   |  design)  |              |  DISABLED       | ------> | provider_key  |  <-
   +-----------+              +-----------------+         | claim_token   |
                                                          +---------------+
   the anon key was never the problem. the absent policy was.

   what leaked            who can revoke it
   session token          Moltbook
   claim token            Moltbook
   provider API key       the person who created the agent   <- not the platform
```

Moltbook launched in late January 2026 as a social network with an unusual
membership rule: the accounts were autonomous AI agents, posting, commenting and
forming communities, with humans watching. Within days a security researcher,
Jameson O'Reilly, found that the whole thing was readable by anyone.

The mechanism is almost disappointingly small. Moltbook ran on **Supabase**, and
the site shipped its Supabase URL and publishable ("anon") key in the client —
which is normal and by design. What was not normal is that **Row-Level Security
was disabled on the tables behind it**. In Supabase, RLS is what turns "this key
identifies the application" into "this key may read this row". Without it, the
anon key is a read-everything key.

So a single query returned every agent's record. Those records held, in
plaintext, in a client-readable table:

- each agent's **secret API key** — spanning OpenAI, Anthropic, AWS, GitHub and
  Google Cloud accounts belonging to the humans who created them,
- claim tokens and verification codes,
- the owner relationships linking every agent back to its creator.

Anyone holding those keys could impersonate any agent on the platform, post as
it, and drive it — without ever failing an authentication check, because they
were authenticating correctly, as the agent.

Two things make this a Function C case study rather than a footnote.

**The blast radius is not the platform.** A social network for agents losing its
own data is a bad day. A social network for agents losing the *provider
credentials of everyone who registered one* is an incident in every one of those
providers' accounts, and the platform cannot revoke them for you.

**The second surface was never touched.** Moltbook's architecture — agents
ingesting and acting on content other agents post — is an indirect prompt
injection surface by construction (A1.3). The breach did not use it. It did not
need to.

The fix was two SQL statements.

> **Sources.** Public reporting on the Moltbook disclosure, late January 2026:
> [Treblle's breakdown](https://treblle.com/blog/moltbook-breach-breakdown),
> [PointGuard AI](https://www.pointguardai.com/ai-security-incidents/moltbook-ai-agent-network-platform-vulnerability),
> [Vectra AI](https://www.vectra.ai/blog/moltbook-and-the-illusion-of-harmless-ai-agent-communities)
> and [Kiteworks](https://www.kiteworks.com/cybersecurity-risk-management/moltbook-ai-agent-security-threat-enterprise-data-protection/).
> Figures below are theirs. Reporting disagrees on the scale — 770,000 agents
> in one account, 1.5 million in another — and both are carried here rather
> than one being chosen.

<svg viewBox="0 0 700 168" width="100%" style="max-width:700px;height:auto;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px"><defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#8A93A6"/></marker></defs><rect x="6" y="16" width="190" height="62" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="101.0" y="44.0" text-anchor="middle" fill="currentColor">browser</text><text x="101.0" y="60.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">anyone, unauthenticated</text><text x="101" y="96" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">ships the Supabase URL</text><text x="101" y="110" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">+ the anon key (by design)</text><rect x="268" y="16" width="170" height="62" rx="5" fill="none" stroke="#E0912F" stroke-width="1.4"/><text x="353.0" y="51.0" text-anchor="middle" fill="#E0912F">Supabase Data API</text><rect x="510" y="6" width="184" height="40" rx="5" fill="none" stroke="#E05C4B" stroke-width="1.4" stroke-dasharray="5 4"/><text x="602.0" y="30.0" text-anchor="middle" fill="#E05C4B">row-level security</text><text x="602" y="60" text-anchor="middle" fill="#E05C4B" font-size="11" font-weight="600">DISABLED</text><rect x="510" y="78" width="184" height="52" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="602.0" y="101.0" text-anchor="middle" fill="currentColor">agents table</text><text x="602.0" y="117.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">every row, every column</text><line x1="197" y1="47" x2="265" y2="47" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="439" y1="47" x2="506" y2="100" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><text x="350" y="150" text-anchor="middle" fill="#8A93A6" font-size="11.5" font-weight="normal">the anon key was never the problem. the absent policy was.</text></svg><div style="font-size:12px;color:#8A93A6;margin-top:2px">RLS is what turns 'this key identifies the application' into 'this key may read this row'. Without it the anon key reads everything.</div>

## 3 · The query, and the two statements that close it

This is worth running rather than drawing, because the interesting part is what comes back — and how little has to change for it to stop.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)"></th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">SQL</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><code>alter table agents enable row level security;</code></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><code>create policy owner_reads on agents for select using (auth.uid() = owner_id);</code></td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Reported as roughly two statements. The gap between an incident and no incident was a policy nobody wrote, not a control nobody could afford.</div>

## 4 · Why the blast radius is not the platform's

Moltbook losing its own data would be a bad day for Moltbook. What was in the table belonged to everyone who had registered an agent, and Moltbook could not revoke any of it.

## 5 · The surface the attack did not need

Worth saying plainly, because it is the part that generalises: the interesting architectural risk in Moltbook was never exercised.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">surface</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">status in this incident</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">where it is taught</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">credential store readable by anyone</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>used — this was the breach</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A3.8, and the Supabase pattern in C2.10</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agents ingest and act on other agents&#x27; posts</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">present, untouched</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A1.3 indirect prompt injection, A1.10 comms poisoning</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agents coordinating at population scale</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">present, untouched</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A1.11, D1.10 fleet correlation</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">An architecture can hold two novel risks and still be undone by a missing row policy. Novelty is not the same as likelihood.</div>

## Your turn

Run `select relname from pg_class where relrowsecurity = false` against your own project, or the equivalent for whatever backend you use. Then find the table holding anything credential-shaped and check whether it is reachable from the client at all — the answer to the second question is the one that decides the size of your bad day.

---

**Next → [C2.10 · Case study — the Supabase pattern: open until closed](https://spbreed.github.io/cyber-commons/lessons/C2.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*